# Customer Churn Analysis – Data Quality Audit and Exploratory Data Analysis

This notebook audits and explores the public IBM Telco Customer Churn sample. It is an **alternative dataset**, not the official EdVyro-provided dataset; the EdVyro resource files were unavailable. No claim of exact compliance with unavailable EdVyro instructions is made.

**Source CSV:** [IBM GitHub raw file](https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv) · [Repository](https://github.com/IBM/telco-customer-churn-on-icp4d)

The notebook preserves the raw file, invokes the documented cleaning/validation pipeline, reviews descriptive results, and displays the checked-in HTML charts.

## 1. Set up and inspect the source

In [1]:
from pathlib import Path
import json
import runpy
import numpy as np
import pandas as pd
from IPython.display import IFrame, display

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(path for path in candidates if (path / "data" / "raw").exists())
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "Telco-Customer-Churn.csv"
print(f"Project folder: {PROJECT_ROOT}")
print(f"Raw source file exists: {RAW_PATH.exists()}")
raw = pd.read_csv(RAW_PATH, dtype={"customerID": "string"}, keep_default_na=False)
print(f"Rows: {raw.shape[0]:,} | columns: {raw.shape[1]}")
display(raw.head())
display(raw.dtypes.rename("dtype").to_frame())

Project folder: /vercel/sandbox/data/customer-churn-analysis
Raw source file exists: True
Rows: 7,043 | columns: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


,dtype
customerID,string
gender,str
SeniorCitizen,int64
Partner,str
Dependents,str
tenure,int64
PhoneService,str
MultipleLines,str
InternetService,str
OnlineSecurity,str


## 2. Data quality checks on the untouched source

In [2]:
# Treat whitespace-only strings as missing for profiling; this does not modify raw on disk.
profile = raw.replace(r"^\s*$", pd.NA, regex=True)
missing = profile.isna().sum().rename("missing_rows").to_frame()
display(missing.loc[missing["missing_rows"].gt(0)])
print(f"Exact duplicate rows: {raw.duplicated().sum():,}")
print(f"Duplicate customer IDs: {raw['customerID'].duplicated().sum():,}")

,missing_rows
TotalCharges,11


Exact duplicate rows: 0
Duplicate customer IDs: 0


In [3]:
# Review unique-value counts and domains before cleaning.
unique_review = pd.DataFrame({
    "unique_values": raw.nunique(dropna=False),
    "examples": [", ".join(map(str, raw[col].drop_duplicates().head(8).tolist())) for col in raw.columns],
})
display(unique_review)

# TotalCharges arrives as numeric text. Blank strings become NaN only in this profiling Series.
total_charges_profile = pd.to_numeric(profile["TotalCharges"], errors="coerce")
print("TotalCharges numeric conversion failures:", int(total_charges_profile.isna().sum()))

,unique_values,examples
customerID,7043,"7590-VHVEG, 5575-GNVDE, 3668-QPYBK, 7795-CFOCW..."
gender,2,"Female, Male"
SeniorCitizen,2,"0, 1"
Partner,2,"Yes, No"
Dependents,2,"No, Yes"
tenure,73,"1, 34, 2, 45, 8, 22, 10, 28"
PhoneService,2,"No, Yes"
MultipleLines,3,"No phone service, No, Yes"
InternetService,3,"DSL, Fiber optic, No"
OnlineSecurity,3,"No, Yes, No internet service"


TotalCharges numeric conversion failures: 11


In [4]:
# IQR flags are a review aid, not proof of bad data. Keep plausible customer records.
numeric_profile = pd.DataFrame({
    "tenure": pd.to_numeric(profile["tenure"], errors="coerce"),
    "MonthlyCharges": pd.to_numeric(profile["MonthlyCharges"], errors="coerce"),
    "TotalCharges": total_charges_profile,
})
rows = []
for column in numeric_profile:
    values = numeric_profile[column].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    rows.append({"measure": column, "q1": q1, "q3": q3, "lower_fence": lower, "upper_fence": upper, "flagged_rows": int(((values < lower) | (values > upper)).sum())})
display(pd.DataFrame(rows).set_index("measure"))

,q1,q3,lower_fence,upper_fence,flagged_rows
measure,,,,,
tenure,9.00,55.0000,-60.00000,124.00000,0
MonthlyCharges,35.50,89.8500,-46.02500,171.37500,0
TotalCharges,401.45,3794.7375,-4688.48125,8884.66875,0


## 3. Cleaning decisions and validation

The pipeline keeps the raw CSV unchanged, trims text, converts `TotalCharges` to numeric, and sets the 11 blank cumulative-charge values to `0.0` only after confirming that each belongs to a zero-tenure customer. No duplicate or IQR-flagged row is removed. The cleaning script contains the full validation checks and writes the cleaned CSV, evidence JSON, report, and chart specifications.

In [5]:
# Re-run the same portable analysis pipeline used to produce the checked-in deliverables.
runpy.run_path(str(PROJECT_ROOT / "scripts" / "analyze_customer_churn.py"))
clean_path = PROJECT_ROOT / "data" / "processed" / "customer_churn_cleaned.csv"
evidence_path = PROJECT_ROOT / "analysis_evidence.json"
clean = pd.read_csv(clean_path)
evidence = json.loads(evidence_path.read_text(encoding="utf-8"))
print(f"Cleaned data: {clean.shape[0]:,} rows × {clean.shape[1]} columns")
display(pd.DataFrame(evidence["quality"]["validation"].items(), columns=["check", "passed"]))

{
  "source_rows": 7043,
  "clean_rows": 7043,
  "source_columns": 21,
  "churned_customers": 1869,
  "churn_rate_pct": 26.537,
  "missing_total_charges": 11,
  "duplicates": 0,
  "validation_passed": true,
  "evidence": "/vercel/sandbox/data/customer-churn-analysis/analysis/analysis_evidence.json",
  "cleaned_csv": "/vercel/sandbox/data/customer-churn-analysis/data/processed/customer_churn_cleaned.csv",
  "report": "/vercel/sandbox/data/customer-churn-analysis/reports/data_quality_report.md",
  "raw_sha256": "16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91"
}
Cleaned data: 7,043 rows × 21 columns


,check,passed
0,row_count_preserved_except_duplicates,True
1,unique_customerID,True
2,no_exact_duplicate_rows,True
3,no_missing_values,True
4,categorical_values_in_expected_domain,True
5,numeric_domain_checks,True
6,churn_labels_are_binary,True
7,charges_and_tenure_nonnegative,True
8,headline_churn_independent_check,True


## 4. Exploratory data analysis

In [6]:
# Numeric descriptive statistics (SeniorCitizen is a binary indicator; interpret accordingly).
display(clean[["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]].describe().T)

churn_summary = clean["Churn"].value_counts().reindex(["No", "Yes"], fill_value=0).rename_axis("Churn").to_frame("customers")
churn_summary["share_pct"] = churn_summary["customers"] / len(clean) * 100
display(churn_summary)

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.00,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.00,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.50,70.35,89.85,118.75
TotalCharges,7043.0,2279.734304,2266.794470,0.00,398.55,1394.55,3786.60,8684.80


,customers,share_pct
Churn,,
No,5174,73.463013
Yes,1869,26.536987


In [7]:
def churn_rate_table(frame, column):
    result = frame.groupby(column, observed=False).agg(
        customers=("customerID", "size"),
        churned=("Churn", lambda values: values.eq("Yes").sum()),
    )
    result = result.loc[result["customers"].gt(0)].copy()
    result["churn_rate_pct"] = result["churned"] / result["customers"] * 100
    return result.sort_values("churn_rate_pct", ascending=False)

display(churn_rate_table(clean, "Contract"))
display(churn_rate_table(clean, "PaymentMethod"))

,customers,churned,churn_rate_pct
Contract,,,
Month-to-month,3875,1655,42.709677
One year,1473,166,11.269518
Two year,1695,48,2.831858


,customers,churned,churn_rate_pct
PaymentMethod,,,
Electronic check,2365,1071,45.285412
Mailed check,1612,308,19.106700
Bank transfer (automatic),1544,258,16.709845
Credit card (automatic),1522,232,15.243101


In [8]:
tenure_order = ["0–12", "13–24", "25–36", "37–48", "49–60", "61–72"]
clean["tenure_band"] = pd.cut(clean["tenure"], bins=[-1, 12, 24, 36, 48, 60, 72], labels=tenure_order)
tenure_results = churn_rate_table(clean, "tenure_band").reindex(tenure_order)
display(tenure_results)

monthly_by_churn = clean.groupby("Churn")["MonthlyCharges"].agg(["count", "mean", "median", "min", "max"])
display(monthly_by_churn)

,customers,churned,churn_rate_pct
tenure_band,,,
0–12,2186,1037,47.438243
13–24,1024,294,28.710938
25–36,832,180,21.634615
37–48,762,145,19.028871
49–60,832,120,14.423077
61–72,1407,93,6.609808


,count,mean,median,min,max
Churn,,,,,
No,5174,61.265124,64.425,18.25,118.75
Yes,1869,74.441332,79.650,18.85,118.35


In [9]:
# Correlation is descriptive. TotalCharges is cumulative and related to tenure by construction.
correlation = clean[["tenure", "MonthlyCharges", "TotalCharges"]].corr(method="pearson")
display(correlation)
print("Do not interpret correlation as causation; TotalCharges is a cumulative measure.")

,tenure,MonthlyCharges,TotalCharges
tenure,1.000000,0.247900,0.826178
MonthlyCharges,0.247900,1.000000,0.651174
TotalCharges,0.826178,0.651174,1.000000


Do not interpret correlation as causation; TotalCharges is a cumulative measure.


## 5. Source-backed visualizations

These three HTML figures were generated from the same analysis evidence and are included under `visuals/`. The first compares churn counts, the second compares contract churn rates, and the switcher contains payment-method churn, tenure-band churn, and mean MonthlyCharges by churn status. Charge values are shown as reported units; no currency is inferred.

In [10]:
chart_files = [
    ("churn_distribution_notion_agent_chart.html", 430),
    ("contract_churn_notion_agent_chart.html", 430),
    ("segment_analysis_notion_agent_chart.html", 570),
]
for filename, height in chart_files:
    path = PROJECT_ROOT / "visuals" / filename
    print(filename)
    display(IFrame(src=f"../visuals/{filename}", width="100%", height=height))

churn_distribution_notion_agent_chart.html


contract_churn_notion_agent_chart.html


segment_analysis_notion_agent_chart.html


## 6. Conclusions and limitations

The cleaned alternative sample contains one row per customer and passes the coded validation checks. The report and evidence JSON provide the actual calculated counts and segment rates. Results are descriptive, do not establish causation, and must not be presented as EdVyro-specific findings. The 11 zero-tenure `TotalCharges` blanks are handled using an explicit assumption; revisit that choice if the official source definition becomes available.